# Транскрипция + диаризация без HuggingFace (openai-whisper + NeMo MSDD)

Запускается в **Google Colab**. Полностью самодостаточный — ничего не импортирует из `pipeline.py`/`pipeline_nemo.py`, весь код в ячейках ниже. Разбит по стадиям: самый нестабильный шаг (диаризация NeMo) можно перезапускать отдельно, не гоняя заново ASR.

**Перед запуском**: Runtime → Change runtime type → Hardware accelerator → GPU (T4 хватает).

## Проверка GPU

Если ниже ошибка/нет вывода про GPU — включите его: Runtime → Change runtime type → GPU.

In [ ]:
!nvidia-smi

## Установка зависимостей

Colab уже поставляется с `ffmpeg` и `torch`+CUDA — ставим только недостающее. `nemo_toolkit` тянет много транзитивных зависимостей, поэтому pip почти наверняка напишет `ERROR: pip's dependency resolver...` про конфликт версий `pandas`/`huggingface-hub`/`fsspec` и т.п. с тем, что предустановлено в Colab под `gradio`/`cudf`/`gcsfs` — это ожидаемо и **не ошибка**: установка при этом завершается успешно, а этот ноутбук ни `gradio`, ни `cudf`, ни `gcsfs` не использует.

Если реально упадёт сам импорт (`import nemo...`, `import whisper`, `from omegaconf import OmegaConf`) — вот тогда Runtime → Restart session и запустите ячейки заново (пакеты и загруженные файлы после restart session сохраняются, переустанавливать/перезаливать не нужно).

In [ ]:
%pip install -q openai-whisper nemo_toolkit[asr] omegaconf pymupdf

## Загрузка аудио и эталонного транскрипта

Загрузите `audio.mp3` (и опционально PDF с эталонным транскриптом для оценки качества) через диалог ниже. Файлы сохранятся в `tmp/` внутри сессии Colab — пропадут при отключении runtime. Для файлов, которые нужны на постоянной основе, удобнее подключить Google Drive (закомментированный вариант в ячейке).

In [ ]:
from google.colab import files
from pathlib import Path

Path("tmp").mkdir(exist_ok=True)
uploaded = files.upload()
for name, content in uploaded.items():
    (Path("tmp") / name).write_bytes(content)

# Альтернатива диалогу загрузки — подключить Google Drive и указать путь к файлам там:
# from google.colab import drive
# drive.mount('/content/drive')
# INPUT_AUDIO = Path('/content/drive/MyDrive/.../audio.mp3')

INPUT_AUDIO = next(Path("tmp").glob("*.mp3"), None) or next(Path("tmp").glob("*.wav"), None)
REFERENCE_PDF = next(Path("tmp").glob("*.pdf"), None)
print(f"Аудио: {INPUT_AUDIO}")
print(f"Эталон: {REFERENCE_PDF}")

## Настройка: импорты, датаклассы, константы, пути

In [ ]:
import difflib
import json
import re
import shutil
import subprocess
import urllib.request
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Tuple


@dataclass
class Word:
    text: str
    start: float
    end: float
    speaker: str
    confidence: float = 1.0


@dataclass
class Turn:
    speaker: str
    role: str
    start: float
    end: float
    text: str
    words: List[Word] = field(default_factory=list)


@dataclass
class SpeakerFeatures:
    speaker: str
    total_duration: float
    turn_count: int
    avg_turn_duration: float
    is_first: bool
    is_last: bool
    question_density: float  # вопросительных конструкций на 100 слов
    teacher_marker_hits: int
    student_marker_hits: int
    word_count: int


ROLE_WEIGHTS = {
    "turn_count": 0.25,
    "inv_avg_turn_duration": 0.15,
    "is_first": 0.15,
    "is_last": 0.10,
    "question_density": 0.25,
    "marker_score": 0.10,
}
MIN_SIGNIFICANT_DURATION_RATIO = 0.03  # спикеры <3% от общего времени речи — фоновая болтовня, не голосуют за роль
MIN_SIGNIFICANT_TURNS = 1

QUESTION_MARKERS = [
    "почему", "как вы думаете", "поясните", "объясните", "согласны ли",
    "в чём", "что если", "каким образом", "расскажите",
]
TEACHER_MARKERS = [
    "давайте", "переходим к", "на этом закончим", "спасибо за доклад", "коллеги",
    "следующий вопрос", "оппонент", "комиссия", "прошу вопросы", "передаю слово",
    "представьтесь", "у нас есть время",
]
STUDENT_MARKERS = [
    "я исследовал", "в своей работе", "целью моей работы было", "хотел бы представить",
    "моя работа посвящена", "в результате исследования", "мой доклад", "я хочу рассказать",
]


def _detect_device() -> str:
    try:
        import torch
        if torch.cuda.is_available():
            return "cuda"
        if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            return "mps"
        return "cpu"
    except ImportError:
        return "cpu"


OUTDIR = Path("output_nemo")
CACHE_DIR = OUTDIR / ".cache"
OUTDIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(exist_ok=True)

WHISPER_MODEL = "large-v3"
LANGUAGE = "ru"
ASR_DEVICE = _detect_device()  # в Colab с GPU-runtime — "cuda"
DIAR_DEVICE = "cuda" if ASR_DEVICE == "cuda" else "cpu"  # NeMo поддерживает только cuda/cpu (не mps)

# Чанкование ASR для очень длинных записей (диаризация остаётся одним проходом на весь файл —
# см. markdown перед Stage 2). Включайте для записей длиннее ~45-60 минут или если Whisper
# упадёт по памяти на всём файле сразу.
CHUNK_LONG_AUDIO = False
CHUNK_TARGET_SEC = 600   # целевая длина куска, сек
CHUNK_MAX_SEC = 720      # жёсткий потолок, если пауза для реза не нашлась

print(f"ASR device: {ASR_DEVICE}, diarization device: {DIAR_DEVICE}")

## Stage 1 — препроцессинг (ffmpeg → 16kHz mono wav)

In [ ]:
def preprocess_audio(input_path: Path, cache_dir: Path) -> Path:
    """
    ffmpeg: любой вход -> 16kHz mono WAV с loudness-нормализацией.

    """
    if shutil.which("ffmpeg") is None:
        raise RuntimeError("ffmpeg не найден в PATH — установите его перед запуском.")
    if input_path is None or not Path(input_path).exists():
        raise RuntimeError(f"Входной аудиофайл не найден: {input_path} — проверьте ячейку загрузки файлов выше.")
    out_path = cache_dir / "audio_16k_mono.wav"
    if out_path.exists():
        return out_path
    cmd = [
        "ffmpeg", "-y", "-i", str(input_path),
        "-ac", "1", "-ar", "16000", "-af", "loudnorm",
        str(out_path),
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode != 0:
        raise RuntimeError(f"ffmpeg упал (код {proc.returncode}). stderr:\n{proc.stderr[-3000:]}")
    return out_path


wav_path = preprocess_audio(INPUT_AUDIO, CACHE_DIR)
wav_path

## Stage 1b — нарезка на чанки для ASR (опционально, `CHUNK_LONG_AUDIO`)

Диаризация NeMo ниже всегда идёт одним проходом на весь файл — она сама справляется с длинными записями через внутреннее скользящее окно, и кросс-чанковую пересборку идентичности спикера городить не нужно. А вот ASR (Whisper) на очень длинных записях может упереться в GPU-память — поэтому именно его можно резать на куски.

Режем **по паузам** (`ffmpeg silencedetect`), не по фиксированному времени — иначе можно разорвать слово или реплику посередине. Если пауза не находится до `CHUNK_MAX_SEC` — режем жёстко на границе (единственный оправданный случай: сплошной монолог длиннее лимита, как раз наш докладчик в тестовом аудио).

In [ ]:
def split_audio_on_silence(
    wav_path: Path, cache_dir: Path, target_sec: float, max_sec: float,
    silence_db: float = -30, min_silence_sec: float = 0.6,
) -> List[Tuple[Path, float]]:
    """
    Режет длинное аудио на куски по паузам. Возвращает [(путь_к_куску, абсолютный_offset_сек), ...].

    """
    import re
    import subprocess as sp

    proc = sp.run(
        ["ffmpeg", "-i", str(wav_path), "-af", f"silencedetect=noise={silence_db}dB:d={min_silence_sec}", "-f", "null", "-"],
        capture_output=True, text=True,
    )
    starts = [float(m) for m in re.findall(r"silence_start:\s*([\d.]+)", proc.stderr)]
    ends = [float(m) for m in re.findall(r"silence_end:\s*([\d.]+)", proc.stderr)]
    silences = list(zip(starts, ends))

    dur_proc = sp.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "csv=p=0", str(wav_path)],
        capture_output=True, text=True,
    )
    total_duration = float(dur_proc.stdout.strip())

    cut_points = [0.0]
    while cut_points[-1] < total_duration - target_sec:
        window_start = cut_points[-1] + target_sec * 0.5
        window_end = min(cut_points[-1] + max_sec, total_duration)
        candidates = [(s + e) / 2 for s, e in silences if window_start <= (s + e) / 2 <= window_end]
        cut = min(candidates, key=lambda c: abs(c - (cut_points[-1] + target_sec))) if candidates else window_end
        cut_points.append(cut)
    cut_points.append(total_duration)

    chunks_dir = cache_dir / "chunks"
    chunks_dir.mkdir(parents=True, exist_ok=True)
    chunks = []
    for i in range(len(cut_points) - 1):
        start, end = cut_points[i], cut_points[i + 1]
        chunk_path = chunks_dir / f"chunk_{i:03d}.wav"
        if not chunk_path.exists():
            sp.run(
                ["ffmpeg", "-y", "-i", str(wav_path), "-ss", str(start), "-to", str(end), "-c", "copy", str(chunk_path)],
                check=True, capture_output=True,
            )
        chunks.append((chunk_path, start))
    return chunks


if CHUNK_LONG_AUDIO:
    audio_chunks = split_audio_on_silence(wav_path, CACHE_DIR, CHUNK_TARGET_SEC, CHUNK_MAX_SEC)
    print(f"Кусков: {len(audio_chunks)}")
    for p, off in audio_chunks:
        print(f"  {p.name}: offset={off:.1f}s")
else:
    audio_chunks = None
    print("CHUNK_LONG_AUDIO=False — обрабатываем весь файл за один проход")

## Stage 2 — ASR (openai-whisper, встроенные word-timestamps)

Кэшируется в `.cache/asr_openai_whisper.json` — при повторном запуске ноутбука эта ячейка не будет гонять модель заново.

In [ ]:
def transcribe_openai_whisper(wav_path: Path, model_name: str, device: str, language: str) -> dict:
    """

    ASR через оригинальный openai-whisper (веса с CDN OpenAI, не с HF), со встроенными

    word-level таймкодами — отдельная forced-alignment модель не нужна.


    Явно освобождает GPU-память модели после транскрипции: следом грузятся три модели NeMo

    (VAD + эмбеддинги + MSDD), и на 15ГБ T4 Whisper large-v3 + NeMo одновременно не помещаются.


    """
    import gc
    import whisper  # openai-whisper — тяжёлая зависимость, импортируется только здесь

    model = whisper.load_model(model_name, device=device)
    result = model.transcribe(str(wav_path), language=language, word_timestamps=True)

    del model
    gc.collect()
    if device == "cuda":
        import torch
        torch.cuda.empty_cache()

    return result


def transcribe_openai_whisper_chunked(
    chunks: List[Tuple[Path, float]], model_name: str, device: str, language: str,
) -> dict:
    """
    Транскрибирует по кускам одной загруженной моделью: initial_prompt для связности
    терминологии между кусками, таймкоды сдвигаются в абсолютные по offset куска.

    """
    import gc
    import whisper

    model = whisper.load_model(model_name, device=device)
    all_segments = []
    prev_tail = ""
    for chunk_path, offset in chunks:
        result = model.transcribe(
            str(chunk_path), language=language, word_timestamps=True,
            initial_prompt=prev_tail or None,
        )
        for seg in result["segments"]:
            seg["start"] += offset
            seg["end"] += offset
            for w in seg.get("words", []):
                w["start"] += offset
                w["end"] += offset
        all_segments.extend(result["segments"])
        prev_tail = result["text"][-200:]

    del model
    gc.collect()
    if device == "cuda":
        import torch
        torch.cuda.empty_cache()

    return {"segments": all_segments}


asr_cache = CACHE_DIR / "asr_openai_whisper.json"
if asr_cache.exists():
    asr_result = json.loads(asr_cache.read_text(encoding="utf-8"))
    print(f"Загружено из кэша: {asr_cache}")
elif CHUNK_LONG_AUDIO:
    asr_result = transcribe_openai_whisper_chunked(audio_chunks, WHISPER_MODEL, ASR_DEVICE, LANGUAGE)
    asr_cache.parent.mkdir(parents=True, exist_ok=True)
    asr_cache.write_text(json.dumps(asr_result, ensure_ascii=False), encoding="utf-8")
else:
    asr_result = transcribe_openai_whisper(wav_path, WHISPER_MODEL, ASR_DEVICE, LANGUAGE)
    asr_cache.parent.mkdir(parents=True, exist_ok=True)
    asr_cache.write_text(json.dumps(asr_result, ensure_ascii=False), encoding="utf-8")

print(f"Сегментов: {len(asr_result['segments'])}")
asr_result["segments"][0]

## Stage 3a — конфиг NeMo MSDD

Схема диаризации NeMo крупная (вложенные `parameters` на VAD/эмбеддингах/кластеризации/MSDD) — собирать её с нуля самодельным словарём ненадёжно (не хватает вложенных ключей, падает с `Missing key parameters`). Правильный путь — взять официальный YAML-темплейт из репозитория NeMo (уже с полными дефолтами) и переопределить только нужные поля.

Файл качается один раз с `raw.githubusercontent.com` (это GitHub, не huggingface.co) и кэшируется локально.

In [ ]:
from omegaconf import OmegaConf

# Зафиксировано под установленную версию (nemo_toolkit 2.7.3) — конфиг проверен, schema подходит
# (diarizer.msdd_model.parameters на месте). Если версия nemo_toolkit в окружении сменится —
# может понадобиться другой тег.
NEMO_DIAR_CONFIG_URL = (
    "https://raw.githubusercontent.com/NVIDIA/NeMo/v2.7.3/examples/speaker_tasks/"
    "diarization/conf/inference/diar_infer_telephonic.yaml"
)

diar_dir = CACHE_DIR / "nemo_diarization"
diar_dir.mkdir(exist_ok=True)

config_path = CACHE_DIR / "diar_infer_telephonic.yaml"
if not config_path.exists():
    urllib.request.urlretrieve(NEMO_DIAR_CONFIG_URL, config_path)

diar_config = OmegaConf.load(config_path)
print(OmegaConf.to_yaml(diar_config.diarizer.msdd_model))  # sanity-check: ключ parameters должен быть на месте

In [ ]:
manifest_path = diar_dir / "manifest.json"
manifest_path.write_text(json.dumps({
    "audio_filepath": str(wav_path.resolve()),
    "offset": 0, "duration": None, "label": "infer",
    "text": "-", "num_speakers": None, "rttm_filepath": None, "uem_filepath": None,
}) + "\n", encoding="utf-8")

diar_config.diarizer.manifest_filepath = str(manifest_path)
diar_config.diarizer.out_dir = str(diar_dir)
diar_config.diarizer.oracle_vad = False
diar_config.diarizer.collar = 0.25
diar_config.diarizer.vad.model_path = "vad_multilingual_marblenet"
diar_config.diarizer.speaker_embeddings.model_path = "titanet_large"
diar_config.diarizer.msdd_model.model_path = "diar_msdd_telephonic"

print(f"manifest: {manifest_path}")
print(f"out_dir: {diar_dir}")

## Stage 3b — запуск диаризации

На CPU на Mac для ~30-минутного аудио может занять заметное время (VAD + speaker embeddings + MSDD — три модели подряд). Если упадёт — правьте и перезапускайте только эту и следующую ячейку, ASR выше уже закэширован.

In [ ]:
from nemo.collections.asr.models import NeuralDiarizer

diarizer = NeuralDiarizer(cfg=diar_config)
diarizer.diarize()

In [ ]:
rttm_path = next((diar_dir / "pred_rttms").glob("*.rttm"))
diarization_segments: List[Tuple[float, float, str]] = []
for line in rttm_path.read_text().splitlines():
    parts = line.split()
    if parts[0] != "SPEAKER":
        continue
    start, dur, speaker = float(parts[3]), float(parts[4]), parts[7]
    diarization_segments.append((start, start + dur, f"SPEAKER_{speaker}"))

print(f"Найдено сегментов: {len(diarization_segments)}")
diarization_segments[:5]

## Stage 4 — слова ASR → спикеры → реплики

In [ ]:
def assign_word_speakers(result: dict, diarization: List[Tuple[float, float, str]]) -> dict:
    """
    Размечает каждое слово ASR спикером по максимальному перекрытию с диаризационным
    сегментом; если слово попало в паузу между сегментами — берёт ближайший по времени.

    """
    for segment in result["segments"]:
        for w in segment.get("words", []):
            w_start, w_end = w["start"], w["end"]
            best_speaker, best_overlap, best_gap, nearest_speaker = "UNKNOWN", 0.0, float("inf"), "UNKNOWN"
            for d_start, d_end, speaker in diarization:
                overlap = min(w_end, d_end) - max(w_start, d_start)
                if overlap > best_overlap:
                    best_overlap, best_speaker = overlap, speaker
                gap = max(d_start - w_end, w_start - d_end, 0.0)
                if gap < best_gap:
                    best_gap, nearest_speaker = gap, speaker
            w["speaker"] = best_speaker if best_overlap > 0 else nearest_speaker
    return result


def group_into_turns(result: dict) -> List[Turn]:
    """
    Разворачивает word-level вывод ASR в последовательные реплики по спикеру.

    """
    words: List[Word] = []
    for segment in result["segments"]:
        for w in segment.get("words", []):
            if "start" not in w or "end" not in w:
                continue  # слово, для которого не нашлось таймкода — пропускаем
            words.append(Word(
                text=w["word"], start=w["start"], end=w["end"],
                speaker=w.get("speaker", "UNKNOWN"), confidence=w.get("probability", 1.0),
            ))

    turns: List[Turn] = []
    for w in words:
        if turns and turns[-1].speaker == w.speaker:
            turns[-1].words.append(w)
            turns[-1].end = w.end
            turns[-1].text += " " + w.text
        else:
            turns.append(Turn(speaker=w.speaker, role="", start=w.start, end=w.end, text=w.text, words=[w]))
    return turns


asr_result = assign_word_speakers(asr_result, diarization_segments)
turns = group_into_turns(asr_result)

print(f"Реплик: {len(turns)}")
for t in turns[:5]:
    print(f"[{t.speaker}] {t.text[:80]}")

## Stage 5 — ролевая классификация (Преподаватель / Студент)

Оценивает не объём речи, а паттерн поведения. Суммарное время речи как единственный признак ломается на защитах/гостевых докладах, где докладчик (студент) говорит большую часть времени, а модератор/экзаменатор (преподаватель) — редко, но направляет ход разговора и задаёт вопросы. Поэтому число реплик, вопросность и позиция в разговоре (кто открыл/закрыл) весят не меньше длительности речи.

In [ ]:
def extract_role_features(turns: List[Turn]) -> Dict[str, SpeakerFeatures]:
    """
    Признаки по каждому спикеру для ролевого скоринга.

    """
    speakers = {t.speaker for t in turns}
    features: Dict[str, SpeakerFeatures] = {}
    for sp in speakers:
        sp_turns = [t for t in turns if t.speaker == sp]
        total_duration = sum(t.end - t.start for t in sp_turns)
        turn_count = len(sp_turns)
        word_count = sum(len(t.words) for t in sp_turns)
        text = " ".join(t.text for t in sp_turns).lower()
        question_hits = text.count("?") + sum(text.count(m) for m in QUESTION_MARKERS)
        teacher_hits = sum(text.count(m) for m in TEACHER_MARKERS)
        student_hits = sum(text.count(m) for m in STUDENT_MARKERS)
        features[sp] = SpeakerFeatures(
            speaker=sp,
            total_duration=total_duration,
            turn_count=turn_count,
            avg_turn_duration=(total_duration / turn_count) if turn_count else 0.0,
            is_first=False,
            is_last=False,
            question_density=(question_hits / word_count * 100) if word_count else 0.0,
            teacher_marker_hits=teacher_hits,
            student_marker_hits=student_hits,
            word_count=word_count,
        )
    if turns:
        features[turns[0].speaker].is_first = True
        features[turns[-1].speaker].is_last = True
    return features


def _normalize(values: Dict[str, float]) -> Dict[str, float]:
    lo, hi = min(values.values()), max(values.values())
    if hi - lo < 1e-9:
        return {k: 0.0 for k in values}
    return {k: (v - lo) / (hi - lo) for k, v in values.items()}


def _split_by_score_gap(ranked: List[Tuple[str, float]]) -> int:
    """
    Сколько спикеров сверху списка считать 'преподавателем' — по наибольшему разрыву в score.

    """
    scores = [s for _, s in ranked]
    gaps = [scores[i] - scores[i + 1] for i in range(len(scores) - 1)]
    return gaps.index(max(gaps)) + 1


def assign_roles(features: Dict[str, SpeakerFeatures]) -> Dict[str, str]:
    total_duration = sum(f.total_duration for f in features.values()) or 1.0
    significant = {
        sp: f for sp, f in features.items()
        if f.total_duration / total_duration >= MIN_SIGNIFICANT_DURATION_RATIO
        and f.turn_count >= MIN_SIGNIFICANT_TURNS
    }
    roles = {sp: "Other" for sp in features}
    if not significant:
        return roles
    if len(significant) == 1:
        only = next(iter(significant))
        roles[only] = "Неизвестно (только один значимый спикер)"
        return roles

    inv_avg = {sp: 1.0 / (f.avg_turn_duration + 1e-6) for sp, f in significant.items()}
    marker = {sp: f.teacher_marker_hits - f.student_marker_hits for sp, f in significant.items()}
    normed = {
        "turn_count": _normalize({sp: f.turn_count for sp, f in significant.items()}),
        "inv_avg_turn_duration": _normalize(inv_avg),
        "question_density": _normalize({sp: f.question_density for sp, f in significant.items()}),
        "marker_score": _normalize(marker),
    }

    scores: Dict[str, float] = {}
    for sp, f in significant.items():
        scores[sp] = (
            ROLE_WEIGHTS["turn_count"] * normed["turn_count"][sp]
            + ROLE_WEIGHTS["inv_avg_turn_duration"] * normed["inv_avg_turn_duration"][sp]
            + ROLE_WEIGHTS["is_first"] * (1.0 if f.is_first else 0.0)
            + ROLE_WEIGHTS["is_last"] * (1.0 if f.is_last else 0.0)
            + ROLE_WEIGHTS["question_density"] * normed["question_density"][sp]
            + ROLE_WEIGHTS["marker_score"] * normed["marker_score"][sp]
        )

    ranked = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
    n_teachers = _split_by_score_gap(ranked)
    teachers, students = ranked[:n_teachers], ranked[n_teachers:]

    for i, (sp, _) in enumerate(teachers, start=1):
        roles[sp] = "Преподаватель" if len(teachers) == 1 else f"Преподаватель {i}"
    for i, (sp, _) in enumerate(students, start=1):
        roles[sp] = "Студент" if len(students) == 1 else f"Студент {i}"
    return roles


def apply_roles(turns: List[Turn], role_map: Dict[str, str]) -> None:
    for t in turns:
        t.role = role_map.get(t.speaker, "Other")


features = extract_role_features(turns)
role_map = assign_roles(features)
apply_roles(turns, role_map)

for sp, role in role_map.items():
    f = features[sp]
    print(f"{sp}: {role}  (речь {f.total_duration:.0f}с, {f.turn_count} реплик, "
          f"вопросов/100слов={f.question_density:.1f})")

## Stage 6 — экспорт (text / srt / json)

In [ ]:
def export_text(turns: List[Turn], path: Path) -> None:
    """
    Текст с ролями, напрямую сравнимый с эталонным script.pdf.

    """
    lines = [f"[{t.role}] {t.text.strip()}" for t in turns]
    path.write_text("\n\n".join(lines), encoding="utf-8")


def _srt_timestamp(seconds: float) -> str:
    ms_total = round(seconds * 1000)
    h, ms_total = divmod(ms_total, 3_600_000)
    m, ms_total = divmod(ms_total, 60_000)
    s, ms = divmod(ms_total, 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


def _split_turn_for_srt(turn: Turn, max_words: int = 14) -> List[Turn]:
    """
    Режет длинную реплику на короткие субтитровые реплики по словам — иначе 20-минутный
    монолог докладчика превращается в один нечитаемый SRT-кадр.

    """
    if len(turn.words) <= max_words:
        return [turn]
    chunks = []
    for i in range(0, len(turn.words), max_words):
        chunk_words = turn.words[i : i + max_words]
        chunks.append(Turn(
            speaker=turn.speaker,
            role=turn.role,
            start=chunk_words[0].start,
            end=chunk_words[-1].end,
            text=" ".join(w.text for w in chunk_words),
            words=chunk_words,
        ))
    return chunks


def export_srt(turns: List[Turn], path: Path) -> None:
    """
    SRT-субтитры, каждый кадр помечен ролью спикера.

    """
    cues = [c for t in turns for c in _split_turn_for_srt(t)]
    lines = []
    for i, cue in enumerate(cues, start=1):
        lines.append(str(i))
        lines.append(f"{_srt_timestamp(cue.start)} --> {_srt_timestamp(cue.end)}")
        lines.append(f"[{cue.role}] {cue.text.strip()}")
        lines.append("")
    path.write_text("\n".join(lines), encoding="utf-8")


def export_json(turns: List[Turn], path: Path, audio_file: str = "", language: str = "") -> None:
    """
    Структурированный JSON под будущий плеер: объект с метаданными файла + список реплик со
    стабильными id и word-level таймкодами (перемотка аудио по клику на реплику/слово, live-
    подсветка текущего слова при воспроизведении по currentTime).

    """
    data = {
        "audio_file": audio_file,
        "language": language,
        "duration": turns[-1].end if turns else 0.0,
        "turns": [
            {
                "id": i,
                "speaker": t.speaker,
                "role": t.role,
                "start": t.start,
                "end": t.end,
                "text": t.text.strip(),
                "words": [
                    {"word": w.text, "start": w.start, "end": w.end, "confidence": round(w.confidence, 3)}
                    for w in t.words
                ],
            }
            for i, t in enumerate(turns)
        ],
    }
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")


export_text(turns, OUTDIR / "transcript.txt")
export_srt(turns, OUTDIR / "transcript.srt")
export_json(turns, OUTDIR / "transcript.json", audio_file=INPUT_AUDIO.name if INPUT_AUDIO else "", language=LANGUAGE)
print(f"Готово: {OUTDIR}")

## Stage 7 — оценка против эталонного `script.pdf`

In [ ]:
def _wer(reference: List[str], hypothesis: List[str]) -> float:
    """
    Word error rate через расстояние Левенштейна на уровне слов.

    """
    n, m = len(reference), len(hypothesis)
    if n == 0:
        return 0.0
    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev, dp[0] = dp[0], i
        for j in range(1, m + 1):
            cur = dp[j]
            dp[j] = prev if reference[i - 1] == hypothesis[j - 1] else 1 + min(prev, dp[j], dp[j - 1])
            prev = cur
    return dp[m] / n


def evaluate_against_pdf(turns: List[Turn], pdf_path: Path) -> None:
    """
    Сравнивает вывод пайплайна с эталонным транскриптом: PDF с ролями в квадратных
    скобках, без таймкодов — поэтому сопоставление текстовое (difflib), а не по времени.

    """
    import fitz  # PyMuPDF — тяжёлая зависимость, импортируется только здесь

    text = "".join(page.get_text() for page in fitz.open(pdf_path))
    parts = re.split(r"\[(Преподаватель|Студент)\]", text)

    ref_words, ref_roles = [], []
    for role, chunk in zip(parts[1::2], parts[2::2]):
        for w in re.findall(r"\w+", chunk.lower()):
            ref_words.append(w)
            ref_roles.append(role)

    hyp_words, hyp_roles = [], []
    for t in turns:
        for w in t.words:
            for tok in re.findall(r"\w+", w.text.lower()):
                hyp_words.append(tok)
                hyp_roles.append(t.role)

    wer = _wer(ref_words, hyp_words)

    matcher = difflib.SequenceMatcher(a=ref_words, b=hyp_words, autojunk=False)
    matched = correct = 0
    for block in matcher.get_matching_blocks():
        for k in range(block.size):
            ref_role, hyp_role = ref_roles[block.a + k], hyp_roles[block.b + k]
            matched += 1
            if ref_role in hyp_role or hyp_role in ref_role:  # "Преподаватель" ~ "Преподаватель 1"
                correct += 1

    print(f"WER: {wer:.1%}  ({len(ref_words)} эталонных слов)")
    if matched:
        print(f"Role accuracy: {correct / matched:.1%}  ({matched} выровненных слов)")
    else:
        print("Role accuracy: недостаточно совпадений текста для оценки")


if REFERENCE_PDF.exists():
    evaluate_against_pdf(turns, REFERENCE_PDF)
else:
    print(f"{REFERENCE_PDF} не найден — пропускаю оценку")